# Notebook 02 — Returns, Risk and Descriptive Analytics

**FIN 4600 · Lab 3 · Financial Data Analytics**

Duran, *Financial Services Technology* (3rd ed.), **Chapter 6 — Data Analytics**

---

Chapter 6 separates analytics into **descriptive** (what happened),
**predictive** (what is likely to happen) and **prescriptive** (what should we
do). This notebook is descriptive analytics: turning a price file into the
risk and performance measures that actually appear on a desk.

You already know these measures from FIN 4000 and FIN 4500. The point here is
not the formula — it is computing it correctly, at scale, from raw data.

**What you will build**

1. Simple and log returns, and why the distinction matters
2. Cumulative growth of a dollar
3. Annualised return, volatility and the Sharpe ratio
4. Rolling volatility, and realised vs implied (VIX)
5. Drawdown
6. The correlation matrix, and what it says about diversification
7. The return distribution, fat tails, and historical Value at Risk
8. Beta, computed two ways

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap


def find_repo_root(start=None):
    """Return the repository root — the folder that contains data/sp500_prices.csv."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "sp500_prices.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root. In VS Code use File > Open Folder "
        "and open the mtu4600-lab3-analytics folder itself, then re-run."
    )


REPO = find_repo_root()
DATA = REPO / "data"
plt.style.use(REPO / "fin4600.mplstyle")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)

TRADING_DAYS = 252  # the conventional number of trading days in a year

In [ ]:
# Rebuild the wide price matrix, so this notebook stands on its own
prices = pd.read_csv(DATA / "sp500_prices.csv", parse_dates=["date"])
companies = pd.read_csv(DATA / "sp500_companies.csv")

close = prices.pivot(index="date", columns="ticker", values="close")
sector_of = companies.set_index("ticker")["sector"].to_dict()

print(f"{close.shape[0]} trading days x {close.shape[1]} tickers")
print(f"{close.index.min().date()} to {close.index.max().date()}")

## 1. Simple returns vs log returns

**Simple return** $R_t = \dfrac{P_t}{P_{t-1}} - 1$. This is what you actually
earn. Simple returns add up *across assets* — a portfolio's return is the
weighted sum of its holdings' simple returns.

**Log return** $r_t = \ln\!\left(\dfrac{P_t}{P_{t-1}}\right)$. Log returns add
up *across time* — a two-day log return is just the sum of the two daily log
returns. They are also closer to symmetric, which most statistical models
assume.

**Use simple returns for portfolio arithmetic; use log returns for
time-series modelling.** Mixing them up is a classic error, and for daily
equity data it is a small one — the two are nearly identical because daily
moves are small. For a stock that halves, it is a very large one.

In [ ]:
simple_ret = close.pct_change()
log_ret = np.log(close / close.shift(1))

comparison = pd.DataFrame(
    {"simple": simple_ret["AAPL"], "log": log_ret["AAPL"]}
).dropna()

print("Daily returns are nearly identical:")
print(comparison.head(3).round(6))
print(f"\nLargest absolute gap across all AAPL days: {(comparison['simple'] - comparison['log']).abs().max():.5f}")

In [ ]:
# The gap grows with the size of the move
demo = pd.DataFrame({"simple_return": [0.01, 0.05, 0.10, 0.50, -0.50, -0.90]})
demo["log_return"] = np.log1p(demo["simple_return"])
demo["difference"] = demo["log_return"] - demo["simple_return"]
demo.round(4)

### Why log returns add across time

In [ ]:
two_day_simple = (1 + simple_ret["AAPL"]).rolling(2).apply(np.prod, raw=True) - 1
two_day_log = log_ret["AAPL"].rolling(2).sum()

check = pd.DataFrame(
    {
        "compounded simple": two_day_simple,
        "summed log": two_day_log,
        "summed log converted back": np.expm1(two_day_log),
    }
).dropna()
check.head(4).round(6)

## 2. Growth of a dollar

The single most useful performance chart. Start with \$1 and compound.

In [ ]:
growth = (1 + simple_ret.fillna(0)).cumprod()

highlight = ["BA", "MSFT", "JNJ", "GE"]

fig, ax = plt.subplots(figsize=(9.5, 5))
for ticker in close.columns:
    if ticker not in highlight:
        ax.plot(growth.index, growth[ticker], color="#d9d8d4", linewidth=1, zorder=1)
for ticker in highlight:
    ax.plot(growth.index, growth[ticker], linewidth=2, label=ticker, zorder=2)

ax.axhline(1.0, color="#52514e", linewidth=0.8, linestyle="--")
ax.set_title("Growth of $1 invested February 2013")
ax.set_ylabel("Value of $1")
ax.set_xlabel("")
ax.legend(loc="upper left")

# Direct labels at the right edge, so identity is never colour-alone
last_date = growth.index[-1]
for ticker in highlight:
    ax.annotate(
        f"  {ticker}  ${growth[ticker].iloc[-1]:.2f}",
        xy=(last_date, growth[ticker].iloc[-1]),
        fontsize=9,
        va="center",
        color="#52514e",
    )
ax.set_xlim(growth.index[0], last_date + pd.Timedelta(days=330))
plt.show()

The grey lines are the other 26 stocks. Showing them de-emphasised gives the
reader context — how the highlighted names did *relative to the field* —
without adding 26 colours nobody can tell apart. Any chart that needs more
than about eight distinguishable colours is the wrong chart.

## 3. Annualising: return, volatility, Sharpe

Daily numbers are annualised by convention:

- mean return: $\times 252$
- volatility: $\times \sqrt{252}$ (variance scales with time, so standard
  deviation scales with its square root)

The square root rule assumes returns are independent across days. They are
not, quite — there is mild autocorrelation and strong volatility clustering —
so treat annualised volatility as a convention, not a truth.

In [ ]:
# Risk-free rate. We only have the 10-year Treasury yield in this repo, so we
# use it as a stand-in. A correct Sharpe ratio uses a SHORT rate (3-month
# T-bills). This approximation overstates the risk-free rate and therefore
# understates every Sharpe ratio below. Flagging an approximation is part of
# the job.
us10y = pd.read_csv(DATA / "us10y_monthly.csv", parse_dates=["date"])
sample_window = us10y[(us10y["date"] >= "2013-02-01") & (us10y["date"] <= "2018-02-28")]
rf_annual = sample_window["yield_pct"].mean() / 100

print(f"Average 10-year Treasury yield over the sample: {rf_annual:.2%}")
print("(Used below as a proxy for the risk-free rate — see the note in the code.)")

In [ ]:
stats = pd.DataFrame(
    {
        "ann_return": simple_ret.mean() * TRADING_DAYS,
        "ann_volatility": simple_ret.std() * np.sqrt(TRADING_DAYS),
    }
)
stats["sharpe"] = (stats["ann_return"] - rf_annual) / stats["ann_volatility"]
stats["sector"] = stats.index.map(sector_of)

stats.sort_values("sharpe", ascending=False).round(3)

### Risk and return together

The classic scatter. Each point is one stock; the eye is looking for whether
more risk actually bought more return over this particular five years.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(
    stats["ann_volatility"],
    stats["ann_return"],
    s=70,
    color="#2a78d6",
    edgecolor="white",
    linewidth=1.5,
    zorder=3,
)
for ticker, row in stats.iterrows():
    ax.annotate(
        ticker,
        (row["ann_volatility"], row["ann_return"]),
        xytext=(0, 9),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        color="#52514e",
    )

ax.axhline(rf_annual, color="#52514e", linewidth=0.8, linestyle="--")
ax.annotate(
    f"risk-free proxy {rf_annual:.1%}",
    xy=(ax.get_xlim()[0], rf_annual),
    xytext=(4, 5),
    textcoords="offset points",
    fontsize=8,
    color="#52514e",
)
ax.set_title("Annualised risk and return, February 2013 – February 2018")
ax.set_xlabel("Annualised volatility")
ax.set_ylabel("Annualised return")
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
ax.grid(axis="x")
plt.show()

### Exercise 1

The chart above covers a single five-year bull market. Recompute `stats` for
calendar year 2015 only — a flat, choppy year — and compare the ranking. How
stable is a five-year Sharpe ratio as a guide to the next year?

In [ ]:
returns_2015 = simple_ret.loc["2015"]

stats_2015 = pd.DataFrame(
    {
        "ann_return": returns_2015.mean() * TRADING_DAYS,
        "ann_volatility": returns_2015.std() * np.sqrt(TRADING_DAYS),
    }
)
stats_2015["sharpe"] = (
    stats_2015["ann_return"] - rf_annual
) / stats_2015["ann_volatility"]
stats_2015["sector"] = stats_2015.index.map(sector_of)

print("2015 ranking by Sharpe ratio:")
print(stats_2015.sort_values("sharpe", ascending=False).round(3).head(10).to_string())
print("\nFive-year ranking by Sharpe ratio:")
print(stats.sort_values("sharpe", ascending=False).round(3).head(10).to_string())

## 4. Rolling volatility, and realised vs implied

A single volatility number for five years hides everything interesting. Risk
is not constant: it clusters. Quiet periods follow quiet periods and violent
ones follow violent ones, which is the empirical fact that the whole GARCH
literature exists to model.

In [ ]:
WINDOW = 60  # roughly a quarter of trading days

rolling_vol = simple_ret.rolling(WINDOW).std() * np.sqrt(TRADING_DAYS)

fig, ax = plt.subplots(figsize=(9.5, 4.5))
for ticker in ["AAPL", "JPM", "JNJ"]:
    ax.plot(rolling_vol.index, rolling_vol[ticker], label=ticker)
ax.set_title(f"{WINDOW}-day rolling annualised volatility")
ax.set_ylabel("Annualised volatility")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
ax.legend()
plt.show()

### Realised vs implied volatility

The VIX is the options market's forecast of S&P 500 volatility over the next
30 days, quoted as an annualised percentage. Our equal-weight basket's
*realised* volatility is the same quantity measured after the fact, in the
same units — so the two belong on one axis.

(A chart with two different y-axes is almost always a mistake. If two series
are not in the same units, make two charts.)

In [ ]:
vix = pd.read_csv(DATA / "vix_daily.csv", parse_dates=["date"]).set_index("date")

basket_ret = simple_ret.mean(axis=1)  # equal-weight basket of our 30 names
realised_21d = basket_ret.rolling(21).std() * np.sqrt(TRADING_DAYS) * 100

vol_compare = pd.DataFrame(
    {"realised (21-day, equal-weight basket)": realised_21d, "VIX (implied, S&P 500)": vix["vix_close"]}
).dropna()

fig, ax = plt.subplots(figsize=(9.5, 4.5))
ax.plot(vol_compare.index, vol_compare.iloc[:, 1], label="VIX (implied)", color="#eb6834")
ax.plot(vol_compare.index, vol_compare.iloc[:, 0], label="Realised (21-day)", color="#2a78d6")
ax.set_title("Implied volatility runs above realised volatility most of the time")
ax.set_ylabel("Annualised volatility (%)")
ax.set_xlabel("")
ax.legend()
plt.show()

gap = (vol_compare.iloc[:, 1] - vol_compare.iloc[:, 0])
print(f"VIX exceeded realised volatility on {(gap > 0).mean():.1%} of days.")
print(f"Median gap: {gap.median():.1f} volatility points.")

That persistent gap is the **variance risk premium** — buyers of options pay,
on average, more than the volatility that subsequently shows up. It is the
reason systematic option-selling strategies make money most of the time and
then occasionally lose several years of gains in a week.

(Note the apples-to-oranges caveat: VIX tracks the S&P 500, our basket is 30
large caps. The direction of the result is robust; the size of the gap is not
precise.)

## 5. Drawdown

Volatility is symmetric. Investors are not — they care about losses. Drawdown
is the peak-to-current decline, and maximum drawdown is the worst of them.
It is usually a better description of what a client actually experienced.

In [ ]:
def drawdown(returns: pd.Series) -> pd.Series:
    """Peak-to-current decline of a cumulative return series."""
    wealth = (1 + returns.fillna(0)).cumprod()
    running_peak = wealth.cummax()
    return wealth / running_peak - 1


dd = close.apply(lambda col: drawdown(col.pct_change()))

max_dd = dd.min().sort_values()
print("Worst maximum drawdowns:")
print((max_dd.head(6) * 100).round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4))
worst = max_dd.index[0]
ax.fill_between(dd.index, dd[worst] * 100, 0, color="#2a78d6", alpha=0.25)
ax.plot(dd.index, dd[worst] * 100, color="#2a78d6")
ax.set_title(f"{worst} drawdown — worst in the sample at {max_dd.iloc[0]:.1%}")
ax.set_ylabel("Drawdown (%)")
ax.set_xlabel("")
plt.show()

### Exercise 2

Add a column to `stats` called `max_drawdown`. Then compute the **Calmar
ratio** — annualised return divided by the absolute value of maximum drawdown
— and rank the stocks by it. Which names move up the ranking compared with the
Sharpe ordering, and why?

In [ ]:
stats["max_drawdown"] = max_dd
stats["calmar"] = stats["ann_return"] / stats["max_drawdown"].abs()

print("Ranking by Calmar ratio:")
print(stats.sort_values("calmar", ascending=False).round(3).to_string())

## 6. Correlation and diversification

Diversification works when assets do not move together. The correlation
matrix is how you check, and it is the input to every portfolio optimiser.

In [ ]:
corr = simple_ret.corr()

# Order the tickers by sector so the block structure is visible
order = sorted(corr.columns, key=lambda t: (sector_of.get(t, "zzz"), t))
corr = corr.loc[order, order]

off_diagonal = corr.where(~np.eye(len(corr), dtype=bool))
print(f"Correlations run from {off_diagonal.min().min():.2f} to {off_diagonal.max().max():.2f}")

### Choosing the colour scale — a two-minute detour that matters

Correlation ranges from −1 to +1 with a meaningful zero, so the textbook
choice is a **diverging** scale: one hue for negative, another for positive,
a neutral grey at zero.

But we just checked, and *every* pair in this sample is essentially
non-negative. Spending half a diverging scale on values that do not occur
leaves every real value washed out. So we use a **sequential** single-hue
scale from 0 to 1, which puts the whole ramp to work.

The general rule: diverging when the data actually straddles a meaningful
midpoint, sequential when it does not. Check first, then choose. (Both
colour maps are built below so you can switch and see the difference.)

In [ ]:
sequential = LinearSegmentedColormap.from_list(
    "fin4600_sequential", ["#ffffff", "#9ec5f4", "#2a78d6", "#0d366b"]
)
diverging = LinearSegmentedColormap.from_list(
    "fin4600_diverging", ["#2a78d6", "#f0efec", "#e34948"]
)

SECTOR_SHORT = {
    "Communication Services": "Comm Svcs",
    "Consumer Discretionary": "Cons Disc",
    "Consumer Staples": "Cons Stpl",
    "Energy": "Energy",
    "Financials": "Financials",
    "Health Care": "Health",
    "Industrials": "Industrials",
    "Information Technology": "Info Tech",
    "Materials": "Materials",
    "Real Estate": "Real Est",
    "Utilities": "Utilities",
}

fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(corr.values, cmap=sequential, vmin=0, vmax=1)
ax.set_xticks(range(len(order)), order, rotation=90, fontsize=8)
ax.set_yticks(
    range(len(order)),
    [f"{t}  ·  {SECTOR_SHORT.get(sector_of.get(t), '?')}" for t in order],
    fontsize=8,
)
ax.set_title("Daily return correlation, tickers grouped by GICS sector")
ax.grid(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label("Correlation", fontsize=9, color="#52514e")
plt.show()

In [ ]:
# Average correlation within a sector vs across sectors
pairs = []
for i, a in enumerate(order):
    for b in order[i + 1 :]:
        pairs.append(
            {
                "a": a,
                "b": b,
                "corr": corr.loc[a, b],
                "same_sector": sector_of.get(a) == sector_of.get(b),
            }
        )
pair_df = pd.DataFrame(pairs)

print(pair_df.groupby("same_sector")["corr"].agg(["count", "mean", "median"]).round(3))
print("\nMost correlated pairs:")
print(pair_df.nlargest(5, "corr")[["a", "b", "corr"]].to_string(index=False))
print("\nLeast correlated pairs:")
print(pair_df.nsmallest(5, "corr")[["a", "b", "corr"]].to_string(index=False))

Same-sector pairs are more correlated than cross-sector pairs — which is
exactly why sector limits exist in portfolio mandates.

The uncomfortable part: correlations are not stable. They rise in crises,
precisely when diversification is supposed to help. Check it.

In [ ]:
rolling_corr = simple_ret["JPM"].rolling(120).corr(simple_ret["BAC"])

fig, ax = plt.subplots(figsize=(9.5, 4))
ax.plot(rolling_corr.index, rolling_corr, color="#2a78d6")
ax.set_title("120-day rolling correlation: JPM and BAC")
ax.set_ylabel("Correlation")
ax.set_xlabel("")
ax.set_ylim(0, 1)
plt.show()

print(f"Range over the sample: {rolling_corr.min():.2f} to {rolling_corr.max():.2f}")

## 7. The distribution of returns, and Value at Risk

Much of finance theory assumes returns are normally distributed. They are
not. They have **fat tails**: extreme moves happen far more often than a
normal distribution predicts. Every risk model that ignored this has
eventually embarrassed someone.

In [ ]:
from scipy import stats as scipy_stats

ticker = "JPM"
r = simple_ret[ticker].dropna()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(r * 100, bins=80, color="#2a78d6", alpha=0.85)

# Overlay the normal distribution with the same mean and standard deviation
x = np.linspace(r.min() * 100, r.max() * 100, 400)
normal_pdf = scipy_stats.norm.pdf(x, r.mean() * 100, r.std() * 100)
ax.plot(x, normal_pdf * len(r) * (r.max() - r.min()) * 100 / 80,
        color="#eb6834", linewidth=2, label="Normal distribution, same mean and SD")

ax.set_title(f"{ticker} daily returns vs a normal distribution")
ax.set_xlabel("Daily return (%)")
ax.set_ylabel("Number of days")
ax.legend()
plt.show()

In [ ]:
print(f"{ticker} daily returns")
print(f"  Mean:               {r.mean():.4%}")
print(f"  Standard deviation: {r.std():.4%}")
print(f"  Skewness:           {r.skew():.3f}   (0 for a normal distribution)")
print(f"  Excess kurtosis:    {r.kurtosis():.3f}   (0 for a normal distribution)")

# How often did moves beyond 3 standard deviations actually happen?
threshold = 3 * r.std()
actual = (r.abs() > threshold).sum()
expected = scipy_stats.norm.sf(3) * 2 * len(r)
print(f"\nDays beyond 3 standard deviations")
print(f"  Predicted by the normal distribution: {expected:.1f}")
print(f"  Actually observed:                    {actual}")
print(f"  Ratio:                                {actual / expected:.1f}x")

### Historical Value at Risk

**VaR at 99% over one day** is the loss that is exceeded on only 1% of days.
It is a quantile of the return distribution, and there are two common ways to
get it:

- **Parametric**: assume normality and use $\mu - 2.326\sigma$. Fast, and
  wrong in the direction that matters.
- **Historical**: take the empirical 1st percentile. Makes no distributional
  assumption, but can only tell you about losses that have already happened.

**Expected Shortfall** (average loss *given* that VaR was breached) is the
measure Basel III moved to, precisely because VaR says nothing about how bad
the tail is.

In [ ]:
position = 1_000_000  # a $1m position

for confidence in (0.95, 0.99):
    historical_var = r.quantile(1 - confidence)
    parametric_var = r.mean() + scipy_stats.norm.ppf(1 - confidence) * r.std()
    expected_shortfall = r[r <= historical_var].mean()

    print(f"{confidence:.0%} confidence, one-day horizon, ${position:,} position in {ticker}")
    print(f"  Historical VaR:     ${-historical_var * position:>10,.0f}   ({-historical_var:.2%})")
    print(f"  Parametric VaR:     ${-parametric_var * position:>10,.0f}   ({-parametric_var:.2%})")
    print(f"  Expected Shortfall: ${-expected_shortfall * position:>10,.0f}   ({-expected_shortfall:.2%})")
    print()

Notice that parametric VaR is the *smaller* number at 99%. Assuming normality
makes the risk look better than it is — which is the one direction you never
want a risk model to be wrong in.

## 8. Beta, two ways

Beta is the sensitivity of a stock to the market. We do not have the S&P 500
index in this repo, so we build an equal-weight basket of our 30 names as a
market proxy. (A real analysis would use a cap-weighted index; the basket is
a teaching stand-in.)

$$\beta_i = \frac{\operatorname{Cov}(R_i, R_m)}{\operatorname{Var}(R_m)}$$

...which is also the slope coefficient of a regression of the stock's returns
on the market's. Computing it both ways is a good habit: if two independent
routes to a number disagree, one of them has a bug.

In [ ]:
market = simple_ret.mean(axis=1).rename("market")

# Route 1: covariance over variance
cov_betas = simple_ret.apply(lambda col: col.cov(market) / market.var())

# Route 2: least-squares regression slope
aligned = simple_ret.dropna()
market_aligned = market.loc[aligned.index]
design = np.column_stack([np.ones(len(market_aligned)), market_aligned.values])
ols_betas = {}
for col in aligned.columns:
    coefficients, *_ = np.linalg.lstsq(design, aligned[col].values, rcond=None)
    ols_betas[col] = coefficients[1]

beta_table = pd.DataFrame(
    {"beta_covariance": cov_betas, "beta_regression": pd.Series(ols_betas)}
)
beta_table["difference"] = beta_table["beta_covariance"] - beta_table["beta_regression"]
beta_table["sector"] = beta_table.index.map(sector_of)

print(f"Largest disagreement between the two methods: {beta_table['difference'].abs().max():.2e}")
beta_table.sort_values("beta_covariance", ascending=False).round(3)

In [ ]:
plot_betas = beta_table["beta_covariance"].sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = ["#eb6834" if b > 1 else "#2a78d6" for b in plot_betas.values]
ax.barh(plot_betas.index, plot_betas.values, color=colors, height=0.7)
ax.axvline(1.0, color="#52514e", linewidth=1, linestyle="--")
ax.annotate(
    "market beta = 1",
    xy=(1.0, len(plot_betas) - 0.3),
    xytext=(4, 0),
    textcoords="offset points",
    fontsize=9,
    color="#52514e",
    va="center",
)
ax.set_title("Beta against an equal-weight basket of the 30 sample stocks")
ax.set_xlabel("Beta")
ax.grid(axis="x")
ax.grid(axis="y", visible=False)
for y, v in enumerate(plot_betas.values):
    ax.text(v + 0.015, y, f"{v:.2f}", va="center", fontsize=8, color="#52514e")
plt.show()

### Exercise 3

Defensive sectors — Utilities, Consumer Staples, Health Care — are supposed
to have betas below 1. Group `beta_table` by sector and check whether that
holds in this sample. Where it does not, can you suggest why? (Hint: this
sample covers 2013–2018, a period with a specific interest-rate story.)

In [ ]:
sector_beta = (
    beta_table.groupby("sector")
    .agg(
        average_beta=("beta_covariance", "mean"),
        minimum_beta=("beta_covariance", "min"),
        maximum_beta=("beta_covariance", "max"),
        number_of_tickers=("beta_covariance", "size"),
    )
    .sort_values("average_beta")
)

print("Average beta by sector:")
print(sector_beta.round(3).to_string())

print("\nDefensive sectors with average beta below 1:")
defensive_sectors = ["Utilities", "Consumer Staples", "Health Care"]
print(sector_beta.loc[sector_beta.index.isin(defensive_sectors), "average_beta"].round(3).to_string())

## 9. Save the outputs

In [ ]:
processed = DATA / "processed"
processed.mkdir(exist_ok=True)

simple_ret.to_csv(processed / "daily_returns.csv")
stats.to_csv(processed / "risk_return_stats.csv")

print("Wrote:")
print("  ", processed / "daily_returns.csv")
print("  ", processed / "risk_return_stats.csv")

---

## Recap

1. Simple returns for portfolio arithmetic; log returns for time-series work.
2. Annualise by 252 for means and $\sqrt{252}$ for volatility — a convention,
   not a law.
3. Volatility clusters. A single number for a five-year period is a summary,
   not a description.
4. Implied volatility sits above realised most of the time; that gap is a
   risk premium, not free money.
5. Correlations rise when you most need them not to.
6. Returns have fat tails. Normal-distribution VaR understates risk.
7. When two routes to the same number disagree, you have a bug.

Everything so far has been **descriptive** — measuring what happened.
Notebook 03 starts building the inputs a **predictive** model needs.

Next: `03_features_and_targets.ipynb`